# NeuroRoute on Colab

Pure-RL PCB router: **thousands of nets, 6-8 layers, learned differential pairs, learned length tuning**.

Read `neuroroute/DESIGN.md` for the architecture and `neuroroute/README.md` for what is already verified.

**This notebook does NOT need the compiled `pcbworld_pns_bridge`.** That build (`notebooks/00_setup.ipynb`) takes ~40 minutes of KiCad-from-source compilation and is only needed for the *old* PNS thread. NeuroRoute's environment is pure PyTorch, and its KiCad validation uses `kicad-cli` from the ordinary apt package.

Run order matters:

1. setup
2. **local verification** - geometry and environment invariants (~2 min, CPU)
3. **the sim-to-real gate** - real KiCad DRC on routed boards. *If this fails, stop.* Training against a lattice KiCad disagrees with produces numbers that mean nothing.
4. baselines - the numbers to beat
5. training

Set **Runtime > Change runtime type > GPU** before step 5.

## 1. Setup

In [ ]:
import os, subprocess, sys

REPO = 'https://github.com/Klutzhehe/Routerv3.git'   # adjust if your remote differs
ROOT = '/content/Routerv3'

if not os.path.isdir(ROOT):
    subprocess.run(['git', 'clone', REPO, ROOT], check=True)
else:
    subprocess.run(['git', '-C', ROOT, 'pull', '--ff-only'], check=False)

os.chdir(ROOT)
sys.path.insert(0, ROOT)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('cpus', os.cpu_count())

In [ ]:
# Checkpoints to Drive. Colab sessions die; `docs/RL_PLAN.md` lists this as
# non-negotiable, and it is the reason `--resume` exists.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT = '/content/drive/MyDrive/neuroroute_checkpoints'
except Exception as exc:
    print('no Drive:', exc)
    CKPT = '/content/neuroroute_checkpoints'
os.makedirs(CKPT, exist_ok=True)
print('checkpoints ->', CKPT)

## 2. Local verification

No GPU, no KiCad. These check the lattice geometry against an independently written brute-force reference, and re-derive the environment's invariants from the occupancy grid rather than trusting its own status flags.

Expected: **all checks pass**. In particular *every net marked done is physically connected* - that one caught the worst bug in the build (two heads writing the same cell in one batched step).

In [ ]:
!python -m neuroroute.scripts.verify_geometry

In [ ]:
!python -m neuroroute.scripts.verify_env

Length tuning, goal 3. Proves vertex drags change routed length, keep the route connected, and restore the board byte-identically when rejected. No meander generator involved - a meander is what alternating drags look like.

In [ ]:
!python -m neuroroute.scripts.verify_refine

## 3. The sim-to-real gate

**The most important cell in this notebook.**

The whole architecture rests on one claim: a lattice whose pitch is `min_track_width + min_clearance` makes cell occupancy equivalent to a clearance check, so anything the fast engine accepts is DRC-clean by construction. This routes real boards with the non-learned baseline, exports them as `.kicad_pcb`, and hands them to **KiCad's own `DRC_ENGINE`**.

Measured locally against KiCad 9.0.2: **0 legality violations over 66 routed nets** at 8 layers with 35% wide traces.

If this reports violations here, **do not train** - fix the pitch or the dilation rule first.

In [ ]:
# The ordinary KiCad package, not a source build. ~2 min.
!apt-get -qq update && apt-get -qq install -y kicad > /dev/null 2>&1
!kicad-cli version

In [ ]:
!python -m neuroroute.scripts.validate_kicad \
    --boards 8 --nets 40 --layers 8 --size 96 --heads 6 \
    --wide-frac 0.35 --keepouts 3 --out /content/drc_out

## 4. Baselines

The numbers a trained policy has to beat. `greedy` walks straight down the geodesic gradient; `layer_hop` adds a via when another layer is closer. Because direction index 0 *is* the gradient direction, an untrained near-zero-init policy behaves like `greedy` - training starts **at** the baseline rather than below it.

In [ ]:
import torch, time, warnings; warnings.filterwarnings('ignore')
from neuroroute.env.baselines import greedy_safe_action, detour_action, layer_hop_action
from neuroroute.env.route_env import EnvConfig, NeuroRouteEnv
from neuroroute.world.engine import WorldConfig
from neuroroute.world.generator import GeneratorConfig
from neuroroute.world.spec import BoardSpec, LayerStack

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

def bench(nets, layers, size, batch, heads, steps, seeds):
    spec = BoardSpec(height_cells=size, width_cells=size, layers=LayerStack(num_layers=layers))
    env = NeuroRouteEnv(EnvConfig(spec=spec,
        world=WorldConfig(batch_size=batch, max_heads=heads, max_nets=max(64, nets+8),
                          max_steps_per_net=steps, device=DEV),
        generator=GeneratorConfig(num_nets=nets, num_components=8),
        max_episode_steps=steps*6))
    for name, fn in (('greedy', greedy_safe_action), ('detour', detour_action), ('layer_hop', layer_hop_action)):
        obs = env.reset(seeds); t0 = time.perf_counter()
        for _ in range(env.cfg.max_episode_steps):
            obs, r, d, i = env.step(fn(obs))
            if bool(d.all()): break
        dt = time.perf_counter() - t0
        st = env.world.board_stats()
        print(f'  {name:>10}: completion {float(env.world.completion().mean()):6.1%}  '
              f'vias {float(st["vias"].float().mean()):5.1f}  {dt:5.1f}s')

seeds = list(range(900000, 900008))
print('20 nets, 2 layers, 64x64');   bench(20, 2, 64,  8, 4,  64, seeds)
print('60 nets, 8 layers, 128x128'); bench(60, 8, 128, 8, 8,  96, seeds)

In [ ]:
# Throughput. This is the number the whole design exists for: every previous
# thread in this repo ran ONE board per process on 2 vCPUs.
spec = BoardSpec(height_cells=128, width_cells=128, layers=LayerStack(num_layers=8))
env = NeuroRouteEnv(EnvConfig(spec=spec,
    world=WorldConfig(batch_size=16, max_heads=8, max_nets=128, max_steps_per_net=96, device=DEV),
    generator=GeneratorConfig(num_nets=60, num_components=8), max_episode_steps=512))
obs = env.reset()
for _ in range(5): obs, *_ = env.step(layer_hop_action(obs))   # warm up
if DEV == 'cuda': torch.cuda.synchronize()
t0 = time.perf_counter(); N = 50
for _ in range(N): obs, *_ = env.step(layer_hop_action(obs))
if DEV == 'cuda': torch.cuda.synchronize()
dt = time.perf_counter() - t0
print(f'{N*16*8/dt:,.0f} routing decisions/sec on {DEV} (B=16, K=8, 8 layers, 128x128)')

## 5. Training

### Stage 0 - plumbing

One net on an empty board. Should reach ~100% quickly. **Anything less is broken plumbing, not a hard problem** - do not move on from a partial result.

In [ ]:
!python -m neuroroute.training.run \
    --stage 0 --device cuda --batch 16 --heads 4 --width 32 \
    --rollout 32 --updates 150 --eval-every 25 \
    --checkpoint-dir {CKPT}/stage0

### Stage 1 - congestion

20 nets on two layers. The bar is the `greedy` number from section 4.

In [ ]:
!python -m neuroroute.training.run \
    --stage 1 --device cuda --batch 16 --heads 8 --width 48 \
    --rollout 32 --updates 800 --eval-every 50 \
    --checkpoint-dir {CKPT}/stage1 --resume

### Stage 3 - eight layers

The first stage where vias are the main lever. This is the capability KiCad's PNS router is **0-for-32** on (`docs/RL_PLAN.md`, Gate A, closed after three sessions), so there is no prior number in this repo to compare against - only the `layer_hop` baseline.

Watch two lines in the log:

* `completion` against the baselines in the eval block - **not** reward. This repo has a measured case of a policy scoring *worse* reward while completing *more* nets.
* `FORECAST GATE`. If the learned occupancy forecast never beats the straight-line demand baseline, the forecaster has learned nothing worth carrying and the latent-rollout stage in `DESIGN.md` section 5 **does not start**. That gate exists so this cannot become negative result #5 by momentum - four previous lookahead efforts here ran well past the point the evidence had answered the question.

In [ ]:
!python -m neuroroute.training.run \
    --stage 3 --device cuda --batch 12 --heads 8 --width 64 \
    --rollout 32 --updates 3000 --eval-every 100 \
    --checkpoint-dir {CKPT}/stage3 --resume

## 6. Look at what it actually routed

A contact sheet of failures shows the failure mode; a reward curve never will. That was the stated reason RL was abandoned here once, and it gets a real answer rather than a shrug.

In [ ]:
import matplotlib.pyplot as plt, numpy as np
from neuroroute.env.baselines import layer_hop_action

spec = BoardSpec(height_cells=96, width_cells=96, layers=LayerStack(num_layers=4))
env = NeuroRouteEnv(EnvConfig(spec=spec,
    world=WorldConfig(batch_size=4, max_heads=6, max_nets=64, max_steps_per_net=96, device='cpu'),
    generator=GeneratorConfig(num_nets=30, num_components=7), max_episode_steps=400))
obs = env.reset(list(range(900000, 900004)))
for _ in range(400):
    obs, r, d, i = env.step(layer_hop_action(obs))
    if bool(d.all()): break

occ = env.world.occ.cpu().numpy()
L = spec.num_layers
fig, axes = plt.subplots(4, L, figsize=(3.0*L, 12))
for b in range(4):
    for l in range(L):
        ax = axes[b, l]
        img = occ[b, l].astype(float)
        ax.imshow(np.where(img > 0, (img % 17) + 3, np.where(img < 0, 1, 0)),
                  cmap='nipy_spectral', interpolation='nearest', vmin=0, vmax=20)
        ax.set_xticks([]); ax.set_yticks([])
        if b == 0: ax.set_title(f'layer {l}', fontsize=10)
        if l == 0: ax.set_ylabel(f'seed {900000+b}\n{float(env.world.completion()[b]):.0%}', fontsize=9)
plt.tight_layout(); plt.show()

## 7. Export a routed board and DRC it

The round trip: lattice -> `.kicad_pcb` -> KiCad's own DRC. Download the file and open it in KiCad to look at the copper.

In [ ]:
from neuroroute.eval.kicad_export import export_board
from neuroroute.scripts.validate_kicad import run_drc, summarise, classify
from pathlib import Path

out = Path('/content/export'); out.mkdir(exist_ok=True)
stats = export_board(env.world, 0, out / 'routed.kicad_pcb')
print(stats)
ok, report = run_drc(out / 'routed.kicad_pcb', out / 'routed.drc.json')
print('DRC readable:', ok)
if ok:
    for group, counts in classify(summarise(report)).items():
        if counts: print(f'  {group}: {counts}')